# Robustness by query type

Compares the reranked Raabta result across all eight frozen development query categories.

**Status:** provisional development evidence; zero locked test queries used.


In [1]:
from pathlib import Path
import csv, hashlib, json, random, statistics
from collections import Counter
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
print(f"Project: {ROOT.name} | fixed seed: {SEED}")


Project: RAABTA_PROJECT_PORTABLE | fixed seed: 20250816


In [2]:
report = json.loads((ROOT / "reports/tables/provisional_robustness.json").read_text(encoding="utf-8"))
assert report["test_queries_used"] == 0
for name, systems in report["query_types"].items():
    values = systems["querybridge_reranked"]
    print(f"{name:30s} n={values['queries']:2d} Recall@10={values['recall_at_10']:.4f}")


abbreviated_roman_urdu         n=15 Recall@10=0.0000
clean_roman_urdu               n=16 Recall@10=0.5000
highly_noisy_roman_urdu        n=15 Recall@10=0.2667
informal_spelling              n=16 Recall@10=0.0625
named_entity                   n=14 Recall@10=0.0714
short_query                    n=15 Recall@10=0.2667
slightly_ambiguous             n=15 Recall@10=0.0667
urdu_english_code_switching    n=14 Recall@10=0.2143


## Per-query-type comparison


In [3]:
rows = []
for query_type, systems in sorted(report['query_types'].items()):
    direct = systems['direct_dense']; bridge = systems['querybridge_no_reranker']; reranked = systems['querybridge_reranked']
    rows.append({'query_type': query_type, 'n': direct['queries'], 'direct_R@10': direct['recall_at_10'], 'bridge_R@10': bridge['recall_at_10'], 'reranked_R@10': reranked['recall_at_10'], 'bridge_delta': round(bridge['recall_at_10'] - direct['recall_at_10'], 6)})
print(json.dumps(rows, indent=2))


[
  {
    "query_type": "abbreviated_roman_urdu",
    "n": 15,
    "direct_R@10": 0.0,
    "bridge_R@10": 0.0,
    "reranked_R@10": 0.0,
    "bridge_delta": 0.0
  },
  {
    "query_type": "clean_roman_urdu",
    "n": 16,
    "direct_R@10": 0.0625,
    "bridge_R@10": 0.3125,
    "reranked_R@10": 0.5,
    "bridge_delta": 0.25
  },
  {
    "query_type": "highly_noisy_roman_urdu",
    "n": 15,
    "direct_R@10": 0.066667,
    "bridge_R@10": 0.266667,
    "reranked_R@10": 0.266667,
    "bridge_delta": 0.2
  },
  {
    "query_type": "informal_spelling",
    "n": 16,
    "direct_R@10": 0.0,
    "bridge_R@10": 0.125,
    "reranked_R@10": 0.0625,
    "bridge_delta": 0.125
  },
  {
    "query_type": "named_entity",
    "n": 14,
    "direct_R@10": 0.0,
    "bridge_R@10": 0.0,
    "reranked_R@10": 0.071429,
    "bridge_delta": 0.0
  },
  {
    "query_type": "short_query",
    "n": 15,
    "direct_R@10": 0.133333,
    "bridge_R@10": 0.333333,
    "reranked_R@10": 0.266667,
    "bridge_delta": 0.2
 

## Weighted summary


In [4]:
total = sum(row['n'] for row in rows)
print({'weighted_direct_R@10': round(sum(row['n'] * row['direct_R@10'] for row in rows) / total, 6), 'weighted_bridge_R@10': round(sum(row['n'] * row['bridge_R@10'] for row in rows) / total, 6), 'weakest_reranked_types': [row['query_type'] for row in sorted(rows, key=lambda row: row['reranked_R@10'])[:3]]})


{'weighted_direct_R@10': 0.091667, 'weighted_bridge_R@10': 0.166667, 'weakest_reranked_types': ['abbreviated_roman_urdu', 'informal_spelling', 'slightly_ambiguous']}


## Interpretation

Outputs above are measurements from frozen local artifacts. Important limitations must remain attached when reused in the report or viva.
